# Exercise 5.3 — Many cubes, rotating about different axes

The vertices of the cube are uploaded to the GPU **only once** (one position buffer, one colour buffer).
To draw 5 cubes we call `drawArrays` 5 times; before each call we send a **different MVP matrix**:

$$M_i = T(\text{position}_i)\; R(\text{speed}_i \cdot t,\ \text{axis}_i), \qquad MVP_i = P\,V\,M_i$$

So the same 36 vertices appear in 5 places, each turning around its own axis
(x, y, z, and two diagonal axes) at its own speed. No new vertices are defined.
The axis is normalised (length 1) first: `glm.rotate` with a longer axis would also stretch the cube.

Order of `T · R`: read right to left — the cube first rotates around its own centre, then it is moved to its place.
(With `R · T` it would orbit around the origin instead.)

## Library
Only **glm-js** is needed (for matrices). The WebGL code is written directly, so this notebook still works
even when the course JS server is down.

In [ ]:
%%html
<script src='https://git.io/glm-js.min.js'></script>

In [ ]:
%%html
<script id="ex53-vs" type="x-shader/x-vertex">
precision mediump float;
attribute vec3 in_Position;
attribute vec3 in_Color;
uniform mat4 MVP;
varying vec3 ex_Color;
void main(void) {
    gl_Position = MVP * vec4(in_Position, 1.0);
    ex_Color = in_Color;
}
</script>

In [ ]:
%%html
<script id="ex53-fs" type="x-shader/x-fragment">
precision mediump float;
varying vec3 ex_Color;
void main(void) {
    gl_FragColor = vec4(ex_Color, 1.0);
}
</script>

In [ ]:
%%html
<script>
// ---------- small WebGL helpers ----------
// compile the two shaders (read from the <script> tags) and link them into a program
window.makeProgram = function (gl, vsId, fsId) {
  const compile = (type, id) => {
    const s = gl.createShader(type);
    gl.shaderSource(s, document.getElementById(id).text);
    gl.compileShader(s);
    if (!gl.getShaderParameter(s, gl.COMPILE_STATUS)) console.error(id, gl.getShaderInfoLog(s));
    return s;
  };
  const p = gl.createProgram();
  gl.attachShader(p, compile(gl.VERTEX_SHADER, vsId));
  gl.attachShader(p, compile(gl.FRAGMENT_SHADER, fsId));
  gl.linkProgram(p);
  return p;
};
// copy a JS array of numbers into a GPU buffer (GL wants 32-bit floats)
window.makeBuffer = function (gl, data) {
  const b = gl.createBuffer();
  gl.bindBuffer(gl.ARRAY_BUFFER, b);
  gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(data), gl.STATIC_DRAW);
  return b;
};
// connect a buffer (3 floats per vertex) to an attribute of the shader
window.useAttrib = function (gl, prog, name, buffer) {
  const loc = gl.getAttribLocation(prog, name);
  if (loc < 0) return;
  gl.bindBuffer(gl.ARRAY_BUFFER, buffer);
  gl.vertexAttribPointer(loc, 3, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(loc);
};
// call render(time) on every animation frame
window.animate = function (render) {
  const step = (time) => { render(time); requestAnimationFrame(step); };
  requestAnimationFrame(step);
};
// ---------- cube: 6 faces x 2 triangles x 3 vertices = 36 vertices ----------
// only coordinates -1 and 1 are used; the 8 corners are repeated for every face
window.cubeData = function () {
  const p = [[-1,-1,-1], [ 1,-1,-1], [ 1, 1,-1], [-1, 1,-1],
             [-1,-1, 1], [ 1,-1, 1], [ 1, 1, 1], [-1, 1, 1]];
  // one different colour per corner (red, green, blue, yellow, ...)
  const c = [[1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,0.5,0], [0.5,0.5,0.5]];
  const faces = [[4,5,6,7], [1,0,3,2], [0,4,7,3], [5,1,2,6], [7,6,2,3], [0,1,5,4]];
  const pos = [], col = [];
  for (const [a, b, cc, d] of faces)
    for (const i of [a, b, cc, a, cc, d]) {      // square face = 2 triangles
      pos.push(...p[i]);
      col.push(...c[i]);
    }
  return { pos, col, n: pos.length / 3 };
};
</script>

In [ ]:
%%html
<canvas id="ex53Canvas" width="500" height="500" style="border:2px solid #000000;"></canvas>
<div id="ex53CanvasInfo" style="font-family:monospace"></div>
<script>
(function () {
  const cv = document.getElementById("ex53Canvas");
  const gl = cv.getContext("webgl");
  const prog = makeProgram(gl, "ex53-vs", "ex53-fs");
  const cube = cubeData();
  const posBuf = makeBuffer(gl, cube.pos), colBuf = makeBuffer(gl, cube.col);   // created ONCE
  const uMVP = gl.getUniformLocation(prog, "MVP");
  gl.enable(gl.DEPTH_TEST);
  gl.clearColor(1, 1, 1, 1);

  // each cube: position + rotation axis + speed. All use the SAME two buffers.
  const cubes = [
    { pos: [-3,  0,  0], axis: [1, 0, 0], speed: 1.0 },
    { pos: [ 0,  0,  0], axis: [0, 1, 0], speed: 0.7 },
    { pos: [ 3,  0,  0], axis: [0, 0, 1], speed: 1.3 },
    { pos: [-1.5, 2.5, -2], axis: [1, 1, 0], speed: 0.9 },
    { pos: [ 1.5, -2.5, -2], axis: [0, 1, 1], speed: 1.6 },
  ];

  const V = glm.lookAt(glm.vec3(0, 3, 12), glm.vec3(0, 0, 0), glm.vec3(0, 1, 0));
  const P = glm.perspective(glm.radians(45), cv.width / cv.height, 0.1, 100);

  animate((time) => {
    gl.clear(gl.COLOR_BUFFER_BIT | gl.DEPTH_BUFFER_BIT);
    gl.useProgram(prog);
    useAttrib(gl, prog, "in_Position", posBuf);
    useAttrib(gl, prog, "in_Color", colBuf);
    for (const c of cubes) {
    const M = glm.translate(glm.mat4(1), glm.vec3(...c.pos))['*'](
              glm.rotate(glm.mat4(1), c.speed * time / 1000, glm.normalize(glm.vec3(...c.axis))));
      gl.uniformMatrix4fv(uMVP, false, P['*'](V['*'](M)).elements);
      gl.drawArrays(gl.TRIANGLES, 0, cube.n);      // same vertices, new matrix
    }
    document.getElementById("ex53CanvasInfo").textContent = cubes.length + " cubes, one vertex buffer";
  });
})();
</script>